### Fraud Detection using Behavioural Risk Modelling & Machine Learning

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Sklearn - data split
from sklearn.model_selection import train_test_split

# Sklearn - preprocessing
from sklearn.preprocessing import StandardScaler

# Sklearn - models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest

# Sklearn - evaluation
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
    classification_report
)

# Sklearn - calibration
from sklearn.calibration import calibration_curve

In [2]:
df2 = pd.read_csv("data/fraud_feature_engineered.csv")
df2.head(1)

,Transaction_ID,Customer_ID,Transaction_Date,Amount,Merchant_Category,Merchant_ID,Card_Type,Transaction_Type,Country,Is_International,...,Customer_Avg_Amount,Amount_vs_Avg,Above_Customer_Avg,No_Chip_No_Pin,High_Distance_And_Amount,International_And_Far,Night_No_Pin,High_Risk_Category,Device_Fraud_Rate,TxnType_Fraud_Rate
0,1,25795,2025-05-28 11:54:36,81.53,Online Services,8459,Gold,POS,Germany,1,...,190.42,0.428159,0,0,0,0,0,1,0.014424,0.015004


In [5]:
features = [
    # Core numeric
    "Amount_log",
    "Hour_of_Day",
    "Distance_From_Home",
    
    # Binary / behavior
    "Is_International",
    "Is_Chip",
    "Is_Pin_Used",
    
    # Engineered risk flags
    "High_Amount_Flag",
    "Is_Night",
    "Is_Weekend",
    "Far_From_Home",
    
    # Customer behavior
    "Customer_Avg_Amount",
    "Amount_vs_Avg",
    "Above_Customer_Avg",
    
    # Security / anomaly
    "No_Chip_No_Pin",
    
    # Interaction features
    "High_Distance_And_Amount",
    "International_And_Far",
    "Night_No_Pin",
    
    # Other Features - appeared weak previously as all ranges similarly
    "High_Risk_Category",
    "Device_Fraud_Rate",
    "TxnType_Fraud_Rate"
]

In [6]:
X = df2[features]
y = df2["Fraud_Flag"]

In [9]:
# Train/Test Split:

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, stratify=y, random_state=42)

In [10]:
X_train.isna().sum()

Amount_log                  0
Hour_of_Day                 0
Distance_From_Home          0
Is_International            0
Is_Chip                     0
Is_Pin_Used                 0
High_Amount_Flag            0
Is_Night                    0
Is_Weekend                  0
Far_From_Home               0
Customer_Avg_Amount         0
Amount_vs_Avg               0
Above_Customer_Avg          0
No_Chip_No_Pin              0
High_Distance_And_Amount    0
International_And_Far       0
Night_No_Pin                0
High_Risk_Category          0
Device_Fraud_Rate           0
TxnType_Fraud_Rate          0
dtype: int64

In [11]:
print("Train fraud rate:", y_train.mean())
print("Test fraud rate:", y_test.mean())

Train fraud rate: 0.015
Test fraud rate: 0.015


In [12]:
# Standardizing Features:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [13]:
# 1: Rule-based Scoring:

df2["Rule_Score"] = (
    0.25 * df2["High_Amount_Flag"] +
    0.20 * df2["Far_From_Home"] +
    0.20 * df2["Above_Customer_Avg"] +
    0.15 * df2["Is_Night"] +
    0.15 * df2["No_Chip_No_Pin"] +
    0.05 * df2["High_Distance_And_Amount"]
)

df2["Rule_Pred"] = (df2["Rule_Score"] >= 0.5).astype(int)

In [14]:
rule_score_test = df2.loc[X_test.index, "Rule_Score"]
rule_pred_test = df2.loc[X_test.index, "Rule_Pred"]

In [17]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("Rule-Based Model")
print("Precision:", precision_score(y_test, rule_pred_test))
print("Recall   :", recall_score(y_test, rule_pred_test))
print("F1 Score :", f1_score(y_test, rule_pred_test))
print("ROC-AUC:", roc_auc_score(y_test, rule_score_test))
print("Brier Score:", brier_score_loss(y_test, rule_score_test))

Rule-Based Model
Precision: 0.015479298769116
Recall   : 0.05533333333333333
F1 Score : 0.024191197901486446
ROC-AUC: 0.49413392893401015
Brier Score: 0.06341297500000001


In [19]:
cm = confusion_matrix(y_test, rule_pred_test)
cm

array([[93221,  5279],
       [ 1417,    83]])